# Run a model ensemble with vmap in_axes

An ensemble of $E$ networks with the same architecture is one stacked set of parameters with a leading axis of size $E$. `jax.vmap` over that axis runs all members at once, and `in_axes` decides whether the input is shared or different per member.

Each member is a two-layer MLP classifier:

$$h = \mathrm{relu}(x W_1 + b_1), \qquad p = \mathrm{softmax}(h W_2 + b_2)$$

Write `ensemble_predict(params, x, per_member)`:

- `params`: dict `{'w1': (E, D, H), 'b1': (E, H), 'w2': (E, H, K), 'b2': (E, K)}`.
- `per_member`: Python bool. If `False`, `x` is `(N, D)` and every member sees the same inputs. If `True`, `x` is `(E, N, D)` and member `e` sees `x[e]` (as in bagging, where each member gets its own batch).
- Return `(probs, mean_probs)`: `probs` is `(E, N, K)`, member `e`'s class probabilities; `mean_probs` is `(N, K)`, their average over members. float32.

Write the forward pass for **one** member with unbatched parameters, then `vmap` it. No Python loop over members.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def ensemble_predict(params, x, per_member):
    """Return (probs (E, N, K), mean over members (N, K))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__dict__": [["w1", {"__nd__": [[[1.0, 0.0], [0.0, 1.0]], [[2.0, 0.0], [0.0, 2.0]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}], ["w2", {"__nd__": [[[1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}]]}, {"__nd__": [[1.0, 0.0], [0.0, 0.0]], "dtype": "float64"}, false], "e": {"__tup__": [{"__nd__": [[[0.7310585975646973, 0.2689414322376251], [0.5, 0.5]], [[0.8807970285415649, 0.11920291185379028], [0.5, 0.5]]], "dtype": "float32"}, {"__nd__": [[0.8059278130531311, 0.1940721720457077], [0.5, 0.5]], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[1.0, 0.0], [0.0, 1.0]], [[2.0, 0.0], [0.0, 2.0]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}], ["w2", {"__nd__": [[[1.0, 0.0], [0.0, 1.0]], [[1.0, 0.0], [0.0, 1.0]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}]]}, {"__nd__": [[[1.0, 0.0]], [[0.0, 1.0]]], "dtype": "float64"}, true], "e": {"__tup__": [{"__nd__": [[[0.7310585975646973, 0.2689414322376251]], [[0.11920291185379028, 0.8807970285415649]]], "dtype": "float32"}, {"__nd__": [[0.4251307547092438, 0.5748692154884338]], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[0.6955197700381686, -0.9794741683587314, -1.5734903329477068, -2.924970571840865, -0.35323216358269055], [1.2476063726111246, 0.03307262346929485, 0.5118440276808229, 1.0232382136634648, -0.8821458480598934], [2.6522866971695453, -0.8769082522563802, 0.3735530621878692, 2.7395808181161874, -0.11425241215164042], [0.11429451370904192, -0.5118795445527522, 0.33452648360440757, -2.1266836963811473, -0.6468087625992227]], [[1.694093202177908, 0.21132613280362944, -0.24039851728557995, 0.372928487050425, 0.4146068139746092], [0.15210326387147993, 0.5785878662409928, -1.6595968004978632, 0.5912552798907551, -0.6735267717733201], [0.7858026370257101, -0.021246697219347518, -0.9981828397738083, -0.07213099306616272, -1.0070801464447374], [1.0127317556552342, -0.028598261237269624, 0.8530917156785938, -1.1150560668236298, 1.6194173395169926]], [[-0.30106130495092354, 1.3652658399724906, 0.8540513447606197, -1.1009451348804589, 0.32315448516353407], [-0.5424503244285709, 0.4093644878806322, 1.43247855108075, 0.6886708352636329, 1.067935889328298], [-1.171569513017854, 0.20394519519921483, 0.3590717720756556, 1.0888977064622498, -0.6194233271487931], [0.5687860933561971, 0.42947130904497827, -1.524318371004195, -1.0397233608240872, 0.4446708098360002]]], "dtype": "float64"}], ["b1", {"__nd__": [[-1.1765680129615403, -1.226916373430967, 0.8952400173622894, 0.28556628961744895, -0.23515623857913578], [0.10549359303397024, -0.40452582539599136, -0.5369294030017739, -0.4605884349903233, 0.38606190863397244], [1.3213036842667867, -0.4707568598432348, -0.06865727792537184, -0.2182780982196439, -1.2600277603013605]], "dtype": "float64"}], ["w2", {"__nd__": [[[0.25194139300863316, 0.8667653893827483, -0.8436849831309196], [0.8330973617910787, -0.5716675976551412, 1.1513005043097746], [1.2158655860944216, -0.383072989490739, -2.0009402393427296], [-0.3479884977165963, -1.3912077061519135, -1.3147207307641426], [-0.2528270425719888, 1.3604499417455416, -0.09210652557839855]], [[-0.219580138267512, -1.8504993469101834, -0.6061244605987609], [-1.3950831707321905, 0.5203277449467695, -0.5282455885197661], [-0.04390937410231113, -0.016708428109463203, -0.18777482468081588], [0.9525678096640364, 0.4867924615107313, -0.31962727095953336], [0.15375911656247143, 0.5527752284525819, 1.3713452319255217]], [[0.12982802652255646, 1.447618942170459, -0.5941755106420729], [-0.23383170493291058, -1.4872048839382854, 0.8218959189256102], [-1.0326085772321911, -1.032463793574626, 0.8420940709449695], [-0.5326586529478279, -0.7531147387143058, 0.9530456660312242], [-0.18430028499434284, 0.9203384889851014, -0.40268257446727856]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.7266658873734084, 1.0785411130489224, 0.08229186612716916], [1.7989992301457534, -0.39614053041389063, -0.21173031778943865], [0.0793709941333931, 1.005520494651668, -0.2875935809547487]], "dtype": "float64"}]]}, {"__nd__": [[-0.4365914827584129, 0.2537373606441843, 1.2498472087876555, -0.416565435932181], [-1.277652251622201, -0.1308513963358957, 0.623763132298492, -0.542791870640899], [0.12424713542015912, -1.8689870874381547, 0.021277643835543144, 0.10273796020804157], [0.6224986114518901, 0.3092743003403864, 0.7717776154755547, -0.9943756131385594], [0.6602886510183212, -1.6512569638236714, -0.5968541812774073, 1.105857448839561], [0.851183379839743, -1.8530139843897926, 0.5884176623353239, -0.43080228767571965]], "dtype": "float64"}, false], "e": {"__tup__": [{"__nd__": [[[0.9996688365936279, 0.0003309666644781828, 1.9725196409581258e-07], [0.999967098236084, 3.274935806985013e-05, 7.771060950290121e-08], [0.07552755624055862, 0.8756012916564941, 0.048871174454689026], [0.8626927137374878, 0.1333174854516983, 0.003989843651652336], [0.24876868724822998, 0.6132686734199524, 0.13796259462833405], [0.07452141493558884, 0.8771852850914001, 0.048293281346559525]], [[0.8030619025230408, 0.08941533416509628, 0.10752272605895996], [0.8030619025230408, 0.08941533416509628, 0.10752272605895996], [0.49715113639831543, 0.09659969061613083, 0.40624916553497314], [0.9561177492141724, 0.017868926748633385, 0.026013344526290894], [0.12757056951522827, 0.005836413241922855, 0.8665930032730103], [0.8516141772270203, 0.015813078731298447, 0.132572740316391]], [[0.007801937870681286, 0.01204665470868349, 0.9801514148712158], [0.05507507547736168, 0.21928006410598755, 0.7256448268890381], [0.017975937575101852, 0.9797210693359375, 0.0023030745796859264], [0.001226722844876349, 0.001775466138496995, 0.9969978332519531], [0.005479759071022272, 0.9941430687904358, 0.0003772657655645162], [0.07947219908237457, 0.896333634853363, 0.02419424057006836]]], "dtype": "float32"}, {"__nd__": [[0.6035109162330627, 0.03393098711967468, 0.36255812644958496], [0.6193680763244629, 0.1029093861579895, 0.2777225375175476], [0.1968848705291748, 0.6506407260894775, 0.15247446298599243], [0.6066790819168091, 0.05098729580640793, 0.34233367443084717], [0.12727300822734833, 0.5377494096755981, 0.3349776268005371], [0.3352026343345642, 0.5964440107345581, 0.06835342198610306]], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.7779212006140349, 0.20445320686653481, 0.12890809713691617, 1.3943221727811774, -1.441975518609771], [-0.4283761626487952, -0.14254520739113477, -1.8406313429052836, 1.1730940844306155, 1.756085887607632], [-0.62511989710298, 0.3815532399707075, -0.26910628695698346, 0.27040122988729115, -0.6755243772111741], [0.831553759483734, -1.1656098487162097, 1.3633889951981013, 0.3563611993886431, 2.424738389106895]], [[-1.6513100001179244, -0.09551857310923698, -1.2855500358708225, -0.1399241221606185, 0.43511420797926476], [-0.9195142527052586, -1.4350772441506012, -1.7337679288327537, 0.22085289280436593, 1.2766907986927296], [1.1718821391201832, 0.03375359913436464, -0.9616653772653441, -0.36329692701267274, -1.507890710749208], [-1.5763462762873433, 1.0659444941812506, -0.27814878318401204, 0.314441516736664, -0.27350567994765285]], [[1.3249204005210136, 1.4375255926601447, -0.7761577399792198, -0.3660937926131038, -0.6752188274196901], [1.3177607937198326, 1.2466016502499169, -0.2902878268806411, 2.299392727793735, 0.02640738413905812], [-1.2159340754805035, -2.0333257583952613, -0.14375006981088342, 0.3213200068496095, -0.5866035100561903], [0.36309552910733295, 1.1950846827570762, -0.05982354325972556, 0.8428941234726923, 1.6645750853020878]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.4977159293553845, -0.5805693055183667, 0.4318041081460906, 0.5366909933858184, 2.1374528604300718], [-0.4345689341851538, -1.5503831767944598, 0.2988020484784649, 0.3257941493500045, -0.4156790739431062], [0.5647275930001924, 0.31822397113899203, 1.2110095061165949, -0.8351099350930211, -0.7024489265594612]], "dtype": "float64"}], ["w2", {"__nd__": [[[-0.02256473387018165, -0.5077870477362139, 1.2710858941464], [0.5791211203916508, -0.022218595716001575, 1.527722135359455], [0.5015889572318717, 0.9994900677226612, 0.8050143701530288], [-0.1474115407619588, -0.14031855353665593, 0.00735619364245958], [-1.0909183334940347, -0.3449326725783397, 0.8542757774328085]], [[0.3097283045736454, -0.9438029600845004, 2.363450803699267], [-1.0301764872720272, 0.6225975377550774, 0.7253857327335095], [1.277256183371252, -0.8594165799580201, 0.024978281793473993], [-0.19065129949129894, 1.147823264572863, 0.39161920988622007], [0.4277717251773925, 0.5518124865452193, 1.9756084968020955]], [[0.6923210417279927, 0.902676961423511, -0.9066326883665152], [0.7231865228015153, -1.2160660955181712, 0.9822591958590071], [0.18981087557973808, -0.839349224009446, -1.0996456465679898], [-0.04155591924191482, 0.17723314196177462, -0.5057705070250724], [-0.2699731640695577, -0.24578202613375505, -0.46647850512526806]]], "dtype": "float64"}], ["b2", {"__nd__": [[-0.12510776464095708, 0.3385733111967336, 1.3675733988457577], [0.7128583362881304, -0.4889460423357985, 1.6988322373240892], [1.2673257922257426, -0.16692565423764538, 1.1095456448541812]], "dtype": "float64"}]]}, {"__nd__": [[[-0.4381431553702777, -0.6564584224008801, -0.6212643461830069, 1.721209632755671], [0.24452045647310966, 0.4235033776376328, 0.17714413404476986, -0.28985087317083824], [0.5998830197226718, 0.7955650311599914, 2.081915887377314, -0.72321589793712], [1.6581453764239278, 0.7966205680332304, -1.5933194436202636, 1.1074729555331235], [1.7491307184740228, 1.2299939066337884, 0.3835589026842411, 0.29981373252204535], [-1.8399895823202774, 0.46211445087811254, 0.3205260965611651, -0.7894413752540872]], [[0.3378620861721728, -0.522935911369819, 2.3901724085950162, -1.0838655772937922], [-0.7711196813260084, -2.0358774000260946, -0.28489777262266996, 0.22182650875085907], [0.11025762964081648, -0.5454539869885834, 0.9655859419905436, 0.32548420115655996], [1.2717606901400118, -0.029405106840989705, -1.6777589875839998, -0.0689826942408083], [-0.935782756107248, 0.8967223429190748, -0.5404817518597566, -0.18255662219160654], [0.03930474516691345, -1.46551992891238, -0.33675471102255655, 0.918293354987768]], [[-0.5262565044562797, -0.12514714210665034, 0.20234335829943753, -0.6383854019834763], [-0.26161620521645224, 0.4681985648028468, -0.3501468098664231, 0.07893265345488432], [-0.11815581619015762, -1.022221767857816, -1.2724182608633312, 1.9966334250594433], [1.3938255429870332, -0.10336301371191282, 0.9643586331830272, 1.1205043265102177], [0.9754855124677695, -0.34737570863682843, -1.4149301193524455, -1.2424443021221736], [-0.22218954270196264, 1.363435395387878, 1.574867005429649, -0.2205970248083793]]], "dtype": "float64"}, true], "e": {"__tup__": [{"__nd__": [[[8.213947744195593e-09, 2.4791465875750873e-06, 0.9999974966049194], [0.006350717972964048, 0.03639109805226326, 0.9572582244873047], [0.04956574738025665, 0.042286280542612076, 0.908147931098938], [2.3275176772585837e-06, 0.00017628558271098882, 0.9998213648796082], [0.001379684661515057, 0.012081597000360489, 0.9865387678146362], [8.301855268655345e-05, 0.0011371193686500192, 0.998779833316803]], [[0.00010137583740288392, 2.0301499148445146e-07, 0.9998984336853027], [0.13304972648620605, 1.3191860261940747e-06, 0.8669489026069641], [0.12624569237232208, 0.01688932627439499, 0.8568649888038635], [0.00618933467194438, 0.003264442551881075, 0.9905462265014648], [0.06533614546060562, 0.021787691861391068, 0.9128761887550354], [0.4212695062160492, 0.005976900923997164, 0.5727536082267761]], [[0.8750820159912109, 0.037585482001304626, 0.08733241260051727], [0.970160722732544, 0.006249829661101103, 0.02358950674533844], [0.983810544013977, 3.500678940326907e-05, 0.016154512763023376], [0.8846593499183655, 0.013754724524915218, 0.1015859842300415], [0.9909060001373291, 0.0009728795848786831, 0.008121109567582607], [0.7831494212150574, 0.15418988466262817, 0.06266070157289505]]], "dtype": "float32"}, {"__nd__": [[0.29172781109809875, 0.012529388070106506, 0.6957427859306335], [0.3698537349700928, 0.01421408262103796, 0.6159322261810303], [0.38654065132141113, 0.019736871123313904, 0.5937225222587585], [0.2969503402709961, 0.005731817334890366, 0.6973179578781128], [0.35254061222076416, 0.011614056304097176, 0.635845422744751], [0.4015006721019745, 0.05376797169446945, 0.5447313785552979]], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.3136588132121814, 1.2203429194657953, 2.0762498634117654], [-0.9003219808281373, 0.49133427637856836, 0.16811858603301996]]], "dtype": "float64"}], ["b1", {"__nd__": [[-0.29625153742449123, -0.2928499145962757, -0.3068388536943865]], "dtype": "float64"}], ["w2", {"__nd__": [[[0.02152207288173195, -1.5092262959174658], [0.024636521891160987, 0.8125416961203998], [0.7807870336863958, 0.8181502040703489]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.35872203290802274, 0.22334381310637721]], "dtype": "float64"}]]}, {"__nd__": [[1.9701508153400376, -0.1468634678981656], [0.1912251455115464, 1.275316380662169], [0.07038936179239937, -1.2574555509307463], [-0.517736982685377, -0.39795172915855714]], "dtype": "float64"}, false], "e": {"__tup__": [{"__nd__": [[[0.16633807122707367, 0.8336619138717651], [0.41998448967933655, 0.5800154209136963], [0.7991641759872437, 0.20083583891391754], [0.6174922585487366, 0.38250765204429626]]], "dtype": "float32"}, {"__nd__": [[0.16633807122707367, 0.8336619138717651], [0.41998448967933655, 0.5800154209136963], [0.7991641759872437, 0.20083583891391754], [0.6174922585487366, 0.38250765204429626]], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[-0.3680234537094524, -1.2510405624195968, 3.0602328613078678, 2.210933322224158, 1.550191988289701, -1.0988731041178257, 3.1694523840550244, 3.0666012868990946], [6.65585599380874, -1.7041735518796957, -0.20816576397638936, 4.25363204758644, -3.1085344474397814, 0.9182002233442923, 0.7878564386101812, 0.29714715703267747], [-2.687902895509437, 0.2540963591928163, -4.304676653687933, 3.693088504343387, -1.32319577998036, 0.5938618284038529, 0.07879678606435433, 0.6796141943102879]], [[0.4822599267906972, 1.3716802112741975, -0.9806481325779282, 0.19369836033256166, -3.213138928568142, -3.0303311498164325, -2.4524240536638877, -2.9355831794989795], [-4.018663701934548, -1.8005171938541902, 1.940689497366983, 1.29881759331681, -1.1771294716518255, 1.0044730126287358, 1.159565806173676, 8.0494415994907], [-4.6690864651706825, 0.8215276128836354, -3.8307716377464818, 3.828944059492322, -0.5578695309488981, 1.4113235241322244, 0.6962688496190288, 2.181150683441506]], [[-8.592138760171588, -3.338934461470271, 0.35342782165456604, -3.8508903484038552, 1.7553309906743613, 3.7883845983489426, -3.7497127325434976, -1.1667170948312864], [1.9874198797823603, 5.525572892656447, 3.1255819588211993, 2.2875967949916527, -2.482173370154512, 0.20636411813108904, 0.49621321683361486, -1.6842286869167784], [1.3515492031521488, -0.8119367683205864, -1.2012482123525616, 6.455909758977415, -0.3838686787712669, 5.691591892789581, -1.0378518816029199, -0.12095111780959843]], [[-3.6119510651380278, -1.6212465478678397, 4.813671660466165, 0.12780926869488582, -5.880355662616164, 4.308687265481078, -2.6020179960759915, -1.2099160532159794], [-0.08321111204499969, -5.578325597563309, 4.632573835124391, 3.8608786918669473, -0.9688282964150663, 1.0731453034636105, -0.8451456335873806, -1.7687044916623016], [-0.6906294599750633, -4.297136431630527, -1.209644229452087, -1.6700415750680655, 5.71044348667282, -4.372189257057894, 3.7738946941954445, 2.1340717089382046]], [[0.4423281707257545, -1.7955426659700375, 5.39497991280557, 4.3517081121023615, -0.12878270716599882, -0.589834988554631, 0.5632924401432091, 5.4509000097050775], [-2.7139329833880543, 0.24822580606148537, 3.3944522683062885, -4.148591424966204, -0.24482725969195218, 3.3285756123063797, -1.3055954973007917, -0.3339872483765306], [-4.619881712289513, 1.1358086613605856, -3.5197881709749987, -1.1650099105128906, 1.0063728214351362, -1.671345842736307, 2.5946947007624424, 2.1325344812274434]]], "dtype": "float64"}], ["b1", {"__nd__": [[0.48111379980704044, 0.977131325591614, -0.16451658339460226, -2.1484044225833374, 0.8566536450705978, 0.7693972612078595, -0.4736260788524117, -0.8367120213190394], [1.1491787299061862, -0.47412067032383715, -0.38698165703347304, -0.44981661343305623, 2.3189136136658206, -0.7250905500124778, -2.59363333735424, -0.14042782491681158], [-1.0763030757146466, -1.8005933853680538, 0.04995795056453464, -0.6180569924835737, 0.7780670234747517, -0.4326708030394698, 1.071601735298032, 0.869112718125672], [-0.08362890458361626, 0.6548380854946209, 1.6841366603616599, 0.5699825502249733, -1.536160440954119, -0.036427954041781285, 0.08934922547190778, 1.124745408193792], [-0.24271709525411891, 1.0031585527203701, -0.24138405280270214, -0.08779150253012741, -0.3500369996713915, -0.10718552775563965, -0.26372293178238154, 0.21546908358184938]], "dtype": "float64"}], ["w2", {"__nd__": [[[-2.6408297928932454, -3.4855447259925194, -0.6429868215149203, 0.04901745188877577], [-1.5998585277814126, 0.7883921093143471, 0.5785459923174119, 4.595276363527611], [-4.7670075672483145, 5.451948609707213, -7.4193558799708175, -5.743968447235422], [0.27673797989738913, -3.370609072468876, -1.4577912934501622, -1.9035414900763143], [-4.490096704058866, -4.682991725200401, -1.7718969951101757, 0.42592356240183227], [1.148797503553193, -0.1376241536313792, -1.6149613063927886, 2.90155369544111], [-7.133162823845011, -0.5639661923983892, 3.126292226410223, 4.148542050041447], [2.938121237891979, 3.37368649270887, 2.4545797747696643, -0.2124590725877118]], [[-7.251950736657196, -2.0972774977453423, -1.4035230508788894, 4.1213793173495645], [-1.8494401879893612, -1.6139843840636787, 1.4020534334570174, -3.4319079462220894], [-0.4407024675848631, 1.1110446451121108, -1.493068111911651, -0.7097074029505797], [-5.813425021739883, -4.226978397174286, -1.2858874872173005, 2.864889901134362], [-2.246928753943882, -3.292905549471535, -2.9337879752711875, 1.4042420893428098], [-4.805052878521382, -0.06032994398511801, -1.6369077020051628, -0.34961152653099997], [-2.7795534627168204, -0.2867315312457466, 3.499634775467083, -1.398319209829769], [-2.0110731114926272, -8.189495427877791, 0.877097813306539, -2.456889151265241]], [[0.6069315460632412, -0.20192641404732992, -2.8878707232194456, -2.4468505113156507], [-4.529811995284308, 5.423443385244182, 2.2673025103505138, 1.9834048049704793], [3.0841361959968454, 3.6716650237536683, 2.107754061985252, 2.617257331077009], [6.964660716877431, -5.520007412258582, 0.2018318202876302, -2.228769669989897], [-0.1863305731720576, -7.865556147428547, -3.5601338440471775, 2.092716999042277], [-4.069664867273438, 1.3387268829399082, -2.776346126769677, -0.7973240984570047], [-4.804862088969281, -2.1222801988131126, -4.778550093881802, 1.372332610390662], [-5.456068737338573, 0.7334498118717526, -1.8618455212454488, -0.7160926296636511]], [[4.077326424886856, -0.9474521436222676, -2.0749030973446256, -0.11293412997273078], [1.9504897361140052, -1.7545933475458892, 4.972431195252281, 0.4265287380331344], [-1.1518098060783872, -2.0247197254341733, -0.5103250740239387, 1.6348723067518705], [3.118192351313615, 0.20438705814652314, -1.7670599323832161, -0.13706420444305523], [-6.28243304966729, 3.3420485674470624, 2.507261706015094, -0.5432043124083431], [2.1488260957546714, -2.866601607060592, 5.5093551420152025, -0.20421259732435137], [-1.0305777021236093, 6.624905775157636, -2.318330732063217, 1.8842337991219167], [-1.8117509353709333, -0.8769590841257846, -0.07625250960309134, -0.9381957054968293]], [[5.172119868574532, 4.092185823559862, -0.8679675480215565, 7.959165254266551], [0.707923468008407, 0.30353800607755027, 2.032347724488308, -2.2525430554672194], [4.837157688579785, -1.9053060135115694, -2.9583757000250404, 3.9380955434924663], [-1.9961303308985954, -0.041573839981190595, -2.9262812836067873, -0.6748126948498876], [0.9236913004558095, -6.5723000136652185, 0.3283605955910008, -1.883691433843392], [4.6715880350697345, 6.516132335094109, 2.029213810412357, 2.205871425591055], [3.5753006427950753, 3.5842238471181678, -3.8800772616829002, -3.56582353737994], [-1.1316082252494764, -3.8499406568555226, -0.7316035457585014, 0.5459791679505721]]], "dtype": "float64"}], ["b2", {"__nd__": [[-0.722730491471366, 0.5291700307229562, 1.4925927126353862, 0.10764998927220501], [0.4847860539546964, 0.01746855320667729, -0.031049442349667285, -0.46501320736978674], [-1.0724837951178625, 0.5975254101831483, 1.113789494959336, -0.010541650758872955], [0.13192158019629, -1.3308774734663014, -0.21254425060206866, 0.3213923834595552], [-0.078193421505025, 0.07212144967351004, -0.9813310277038408, -1.2490932046968188]], "dtype": "float64"}]]}, {"__nd__": [[-0.24035655914051568, 1.4773257579465242, -0.37522520311689994], [-1.47943133029661, -0.6512056566460908, -0.20745136569617995]], "dtype": "float64"}, false], "e": {"__tup__": [{"__nd__": [[[8.571980880103455e-14, 2.3704779770222072e-20, 1.1325349191793066e-07, 0.9999998807907104], [1.1802544381547212e-14, 4.3429246310289216e-11, 5.223630555883929e-11, 1.0]], [[4.70037488206794e-14, 0.0, 1.0, 7.082273136005401e-13], [3.0545321756552e-38, 1.0241382079799919e-26, 1.1644207110504227e-26, 1.0]], [[1.4223604836757645e-30, 1.0, 1.4187546474495405e-18, 2.78409292148174e-11], [0.0004615437937900424, 0.17094303667545319, 4.0920549939755266e-19, 0.8285953998565674]], [[0.9999080896377563, 1.0097764030005897e-19, 1.3764121136253249e-12, 9.196012251777574e-05], [7.884056630589396e-22, 3.8056627090554684e-05, 0.9999619722366333, 3.7473228865639654e-14]], [[1.0, 1.7597417967252227e-10, 5.685757257324962e-23, 7.512364341710764e-11], [0.95524662733078, 0.041193656623363495, 0.000465077580884099, 0.003094640327617526]]], "dtype": "float32"}, {"__nd__": [[0.39998161792755127, 0.20000000298023224, 0.20000003278255463, 0.20001836121082306], [0.19114163517951965, 0.042434949427843094, 0.20008540153503418, 0.5663380026817322]], "dtype": "float32"}]}}, {"a": [{"__dict__": [["w1", {"__nd__": [[[1.192959497469517, 1.062917305599356], [-0.7963200584994627, -1.001136598266466], [-0.26016183912163426, -0.018856032041332172], [-0.9962247724623505, 0.9363133768567675], [-0.44794164071604414, 0.2310534922529917], [-0.21246331701873045, 0.48502368643186405]], [[-0.13682193276385876, 0.21983507484239379], [1.18890810539395, 0.20364462973241249], [1.3977049002988637, -1.2777861700271238], [-0.2250572346925188, -1.0629111362851877], [-1.438769693918155, -0.08063085529417531], [1.8346031021605052, 0.19602734551259063]], [[0.9435205938322058, -1.9327789576162204], [0.1785324634680391, -1.185713929752006], [0.16699579200121295, 0.9532572432445768], [-0.5389687262788927, 0.999665590495049], [-0.8481879451310014, 0.02258527378067816], [-0.7099578905995387, -0.010600901477116923]], [[-0.25466529778401376, -2.2681420674593933], [-0.46480917826853807, -0.6045658712263449], [1.542868194624471, -2.582515029685851], [-0.045312095597929425, 0.6814962556100485], [1.7458484913524908, 1.1569067804070994], [-0.17288631984177252, 1.3000050771385196]]], "dtype": "float64"}], ["b1", {"__nd__": [[-0.2542775636969874, -0.3130792436344003], [-0.4664873032534413, -0.5012903082243831], [1.3711895349110803, -1.752471211728634], [-1.6657734642242692, 0.7935912533053499]], "dtype": "float64"}], ["w2", {"__nd__": [[[0.31456084140577967, 0.6954957621096794, -0.40377730365148473, 0.4972367052526234, -0.3991956222630558], [-0.668874794079826, -0.60221853948807, -0.03895116516373576, -0.11039403223470962, -0.5712307251536914]], [[1.378631629657112, 0.2556029855529457, -0.5313461299111182, -1.486591566965085, -0.26888759140703106], [-1.8054048173087167, -1.0804994932658816, -1.338835401702493, -1.086495638508617, -0.04160543417235176]], [[-3.424861920192112, 1.2016136475120398, 0.1838248821737053, -0.691472266738389, -0.8712969588681821], [0.446923950911591, -0.38261040581803674, 1.445686476730443, -0.41848066423945357, -1.1934436955501484]], [[-1.6850557199100318, -0.3729128334469439, 0.7443552507995297, -1.4828711946209745, 0.9779835668132821], [0.5547847889544306, -0.5854866171834536, 0.9956891580991444, -2.000558271774936, -2.4374547025430147]]], "dtype": "float64"}], ["b2", {"__nd__": [[0.0010952950627472952, 0.14377947513977832, 0.10068661053103783, -0.31203218165186963, -1.7237080886257876], [0.27461666481288644, -0.6236423290008428, 0.6009752973693873, -0.05395273453720848, -1.4168416620166757], [0.338894323981478, 0.5350136896198718, 0.5279734051047128, -0.9610094138696536, -1.29237159359152], [0.5310442818062122, -0.3171569192501934, -0.3734881580256934, 0.7374219093881335, 0.7918720115146214]], "dtype": "float64"}]]}, {"__nd__": [[[0.2500893491406277, -0.46935254411713384, 1.2425439406407037, -1.2811908616096912, -0.2529918729733742, 0.07263683359317463]], [[-1.6196545080299112, -0.9628131408754413, 0.6240633453960488, -0.32874800114636177, -1.3000339812243762, 1.4150090551027943]], [[-0.0559691411456967, -0.13595478637364397, 1.5845822234720024, -1.7058336617708536, -0.7702093115895071, -1.1493689320696276]], [[1.0830160125016657, -1.0472731332620961, 2.1189999065975886, 0.24718026656422393, 0.07680220425248857, 0.7481886575387039]]], "dtype": "float64"}, true], "e": {"__tup__": [{"__nd__": [[[0.22616992890834808, 0.4564555287361145, 0.08698859065771103, 0.21625860035419464, 0.014127341099083424]], [[0.9946962594985962, 0.004419954959303141, 0.0006343068671412766, 7.060838015604531e-06, 0.00024240519269369543]], [[9.456424088227777e-09, 0.9822779893875122, 0.017552746459841728, 0.00012506455823313445, 4.41515221609734e-05]], [[0.004995402414351702, 0.0229373499751091, 0.16345490515232086, 0.008850379846990108, 0.799761950969696]]], "dtype": "float32"}, {"__nd__": [[0.30646538734436035, 0.3665226995944977, 0.06715764105319977, 0.05631027743220329, 0.20354396104812622]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(ensemble_predict, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")